# RCCN Pipeline：运行主函数并查看结果

这个 notebook 是第一版 Python RCCN pipeline 的轻量入口。它展示主函数在哪里、如何修改参数、如何运行一个小模拟，以及如何读取和展示保存后的输出。

默认情况下，它只读取已有输出，不会启动长时间模拟。需要重新运行模拟时，把 `RUN_SIMULATION = True`。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

PROJECT_ROOT

In [ ]:
from rccn_persistence.config import make_debug_params, make_default_params, make_output_paths
from rccn_persistence.io_utils import ensure_output_dirs, save_params, save_simulation_outputs, load_simulation_outputs
from rccn_persistence.simulation import run_batch
from rccn_persistence.spin_analysis import run_spin_clustering_analysis
from rccn_persistence.observables import compute_survival_by_Tw

paths = make_output_paths(PROJECT_ROOT)
ensure_output_dirs(paths)
paths

## 1. 选择参数

学习和调试时先用 `make_debug_params()`。如果要做更大的科学运行，再使用 `make_default_params()`，并谨慎调整 `n_runs` 和 `waiting_times`。

In [ ]:
params = make_debug_params()

# 用于 notebook 快速检查的 tiny 参数。
# 如果要运行正常 debug 设置，可以删掉这些 overrides。
params.update({
    "num_spins": 64,
    "n_runs": 2,
    "waiting_times": [3, 5],
    "init_time": 20,
    "relax_time": 120,
    "early_recovery_delta": 10,
    "baseline_window": 10,
})

params

## 2. 可选：运行模拟

如果要重新运行模型并覆盖 `output/rccn_simulation/` 中的当前文件，把 `RUN_SIMULATION = True`。

In [ ]:
RUN_SIMULATION = False

if RUN_SIMULATION:
    save_params(params, paths["simulation"])
    batch_result = run_batch(params)
    save_simulation_outputs(batch_result, paths)
    print(f"Saved simulation outputs to: {paths['simulation']}")
else:
    print("Simulation skipped. Reading existing outputs if they exist.")

## 3. 读取模拟输出

In [ ]:
simulation_data = load_simulation_outputs(paths)

metadata = simulation_data["metadata"]
magnetization = simulation_data["magnetization"]
spin_release = simulation_data["spin_release"]

print("metadata:", metadata.shape)
print("magnetization:", magnetization.shape)
print("spin_release:", spin_release.shape)
metadata.head()

## 4. 查看 Magnetization

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
for (Tw, run_id), group in magnetization.groupby(["Tw", "run_id"]):
    ax.plot(group["time"], group["magnetization"], alpha=0.7, label=f"Tw={Tw}, run={run_id}")

ax.set_xlabel("time")
ax.set_ylabel("cycle-averaged magnetization")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()

## 5. 从保存的 Spin Snapshots 运行 PCA / Clustering

In [ ]:
analysis_result = run_spin_clustering_analysis(simulation_data, params)

analysis_result["pca_scores"].to_csv(paths["clustering"] / "pca_scores.csv", index=False)
analysis_result["explained_variance"].to_csv(paths["clustering"] / "pca_explained_variance.csv", index=False)
analysis_result["cluster_labels"].to_csv(paths["clustering"] / "cluster_labels.csv", index=False)
analysis_result["cluster_occupancy_by_Tw"].to_csv(paths["clustering"] / "cluster_occupancy_by_Tw.csv", index=False)
analysis_result["recovery_time_by_cluster"].to_csv(paths["clustering"] / "recovery_time_by_cluster.csv", index=False)

analysis_result["cluster_occupancy_by_Tw"]

## 6. 绘制 PCA Scores

In [ ]:
pca_scores = analysis_result["pca_scores"]
cluster_labels = analysis_result["cluster_labels"]

plot_df = pca_scores.merge(metadata, on="run_id").merge(cluster_labels, on="run_id")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
scatter0 = axes[0].scatter(plot_df["PC1"], plot_df["PC2"], c=plot_df["Tw"], s=50)
axes[0].set_title("colored by Tw")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
fig.colorbar(scatter0, ax=axes[0], label="Tw")

scatter1 = axes[1].scatter(plot_df["PC1"], plot_df["PC2"], c=plot_df["cluster"], s=50)
axes[1].set_title("colored by cluster")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
fig.colorbar(scatter1, ax=axes[1], label="cluster")

fig.tight_layout()

## 7. 根据 Recovery Times 绘制 Survival Curves

In [ ]:
survival = compute_survival_by_Tw(metadata, params["relax_time"])

fig, ax = plt.subplots(figsize=(6, 4))
for Tw, group in survival.groupby("Tw"):
    ax.plot(group["time"], group["survival"], label=f"Tw={Tw}")
ax.set_xlabel("recovery time")
ax.set_ylabel("survival")
ax.legend(frameon=False)
fig.tight_layout()